In [1]:
from google.colab import files

In [2]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/aykahsay/Data-Mining/main/data/raw/raw.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Clean 'Description'

In [3]:
count = df['Description'].isna().sum() + df['Description'].dropna().astype(str).str.startswith('?').sum()
count

np.int64(1521)

In [4]:
import pandas as pd

# Step 1: Identify strange descriptions (NA, empty, or starts with '?')
strange_mask = df['Description'].isna() | df['Description'].str.strip().eq("") | df['Description'].str.strip().str.startswith("?")
df['StrangeDesc'] = strange_mask

# Step 2: Compute mode descriptions for each StockCode (excluding strange ones)
valid_descriptions = df[~df['StrangeDesc']]
stockcode_to_mode_desc = valid_descriptions.groupby('StockCode')['Description'].agg(lambda x: x.mode()[0] if not x.mode().empty else pd.NA)

# Step 3: Impute or mark for drop
def impute_or_drop(row):
    if row['StrangeDesc']:
        mode_desc = stockcode_to_mode_desc.get(row['StockCode'], pd.NA)
        return mode_desc if pd.notna(mode_desc) else pd.NA
    return row['Description']

df['Description'] = df.apply(impute_or_drop, axis=1)

# Step 4: Drop rows where Description is still missing after attempted imputation
df.dropna(subset=['Description'], inplace=True)

# Step 5: Drop helper column
df.drop(columns='StrangeDesc', inplace=True)
descCleanDf = df.copy()

In [5]:
count = descCleanDf['Description'].isna().sum() + descCleanDf['Description'].dropna().astype(str).str.startswith('?').sum()
count

np.int64(0)

## Clean 'Quantity'

In [6]:
(descCleanDf['Quantity'] < 0).sum()

np.int64(10527)

In [7]:
quantCleanDf = descCleanDf.copy()
quantCleanDf['Quantity'] = quantCleanDf['Quantity'].abs()
(quantCleanDf['Quantity'] < 0).sum()

np.int64(0)

## Clean 'UnitPrice

In [8]:
 (quantCleanDf['UnitPrice'] == 0).sum()

np.int64(2403)

In [9]:
import pandas as pd
import numpy as np

# Make a copy to work on
df_cleaned = quantCleanDf.copy()

# Mask for UnitPrice == 0
zero_price_mask = df_cleaned['UnitPrice'] == 0
zero_price_rows = df_cleaned[zero_price_mask]

# Group modes
valid_prices = df_cleaned[df_cleaned['UnitPrice'] > 0]

mode_by_stock_country = valid_prices.groupby(['StockCode', 'Country'])['UnitPrice'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
mode_by_stock = valid_prices.groupby('StockCode')['UnitPrice'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)

# Imputation function
def impute_price(row):
    key = (row['StockCode'], row['Country'])
    if key in mode_by_stock_country:
        return mode_by_stock_country[key]
    elif row['StockCode'] in mode_by_stock:
        return mode_by_stock[row['StockCode']]
    else:
        return np.nan  # Will be dropped

# Apply and convert to float
imputed_prices = zero_price_rows.apply(impute_price, axis=1).astype(float)

# Assign imputed values safely
df_cleaned.loc[zero_price_mask, 'UnitPrice'] = imputed_prices

# Drop rows with missing or zero prices
unitPriceCleanDf = df_cleaned[df_cleaned['UnitPrice'].notna() & (df_cleaned['UnitPrice'] > 0)].copy()

# Confirm any zeroes left
print("Remaining UnitPrice == 0:", (unitPriceCleanDf['UnitPrice'] == 0).sum())


Remaining UnitPrice == 0: 0


## Clean 'CustomerID'

In [10]:
unitPriceCleanDf["CustomerID"].isna().sum()

np.int64(134944)

In [11]:
# Create a mapping from InvoiceNo to the most common CustomerID (mode)
invoice_customer_mode = (
    unitPriceCleanDf[unitPriceCleanDf['CustomerID'].notna()]
    .groupby('InvoiceNo')['CustomerID']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

# Copy the DataFrame
df_cleaned = unitPriceCleanDf.copy()

# Fill missing CustomerID using the mapping
df_cleaned['CustomerID'] = df_cleaned.apply(
    lambda row: invoice_customer_mode[row['InvoiceNo']] if pd.isna(row['CustomerID']) and row['InvoiceNo'] in invoice_customer_mode else row['CustomerID'],
    axis=1
)

# Drop rows where CustomerID is still missing after attempt to impute
custIDcleanDf = df_cleaned[df_cleaned['CustomerID'].notna()].copy()


In [12]:
custIDcleanDf["CustomerID"].isna().sum()

np.int64(0)

## Clean 'Country'

In [13]:
(custIDcleanDf["Country"] == "Unspecified").sum()

np.int64(244)

In [14]:
# Step 0: Work on a copy of the original dataframe to avoid warnings
df_cleaned = custIDcleanDf.copy()

# Step 1: Lowercase the Country column for consistency
df_cleaned['Country_clean'] = df_cleaned['Country'].str.lower()

# Step 2: Create a mapping from CustomerID → most frequent valid Country
customer_country_mode = (
    df_cleaned[df_cleaned['Country_clean'] != 'unspecified']
    .groupby('CustomerID')['Country']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

# Step 3: Fill 'Unspecified' countries using the mapping
mask_unspecified = df_cleaned['Country_clean'] == 'unspecified'
mask_customer_valid = df_cleaned['CustomerID'].notna()

# Apply mapping
df_cleaned.loc[mask_unspecified & mask_customer_valid, 'Country'] = (
    df_cleaned.loc[mask_unspecified & mask_customer_valid, 'CustomerID']
    .map(customer_country_mode)
)

# Step 4: Drop rows where Country is still 'unspecified' or missing
df_cleaned = df_cleaned[df_cleaned['Country'].str.lower() != 'unspecified']
df_cleaned = df_cleaned[df_cleaned['Country'].notna()]

# Step 5: Drop helper column safely
df_cleaned.drop(columns='Country_clean', inplace=True)

# Optional: overwrite the original df if needed
df_final = df_cleaned.copy()

In [15]:
(df_final["Country"] == "Unspecified").sum()

np.int64(0)

In [16]:
# Save to a CSV file
df_final.to_csv("transformed.csv", index=False)